# Deep Dive into LLMs - Multi- Agents




#### THIS SECTION OF CODE IS COVERED IN THE EARLIER CLASS. BRIEF REVISION & SET UP THE GROUND FOR THE MULTI-AGENTIC AI POWERED LLMs.
-----------------------------------------------------------------------------

*A basic chatbot that can answer customer queries*

## Problem Statement
Shopping online can be overwhelming. You search for a simple pair of shoes, but end up scrolling through countless options—many irrelevant, some too expensive, others just not right. Traditional search engines rely on keywords, often missing what you truly need.

Let's build an **AI-powered product discovery chatbot** changes this. Using **advanced language models and vector-based search**, it goes beyond keywords to understand your intent, offering **personalized, context-aware recommendations in seconds**.

<center><img src="https://www.pranathiss.com/static/assets/images/ai-powered-chatBot.webp" width=500/></center>

This smart solution enhances the shopping experience, increasing customer satisfaction, engagement, and conversions. The future of e-commerce is here—smarter, intuitive, and built for you.

### What we did already??
- Conversational chatbot that answers basic queries & is made contextually aware with the help of RAG & Vector Stores.


### What are we going to do today?
- Add agents to the chatbot to enhance its features.


In [ ]:
# Installing required packages
!pip install langchain langchain-community langchain-core langchain-openai faiss-cpu langgraph langchain-text-splitters anthropic langchain-anthropic

!pip install beautifulsoup4
!pip install tavily-python
!pip install gradio


# Necessary Imports
import kagglehub
import csv
import pandas as pd
import math
import numpy as np
import os
import getpass
from langchain_core.output_parsers import StrOutputParser
from google.colab import drive

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
!pip install -U langchain-openai

In [ ]:
# Verify that the import works correctly now
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Import successful!")

In [ ]:
!pip install -U langchain-core langchain-openai

In [ ]:
!pip install --force-reinstall langchain-core langchain-openai

In [ ]:
# Setting up Open AI key
os.environ["OPENAI_API_KEY"] = getpass.getpass()

In [ ]:
# Quick check and request for OpenAI API Key if not set
import os
import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Please enter your OpenAI API Key: ")
else:
    print("✅ OpenAI API Key is already set in your environment!")

In [ ]:
# Loading the extracted dataset and setting up the RAG system
import os
import pandas as pd
import math
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

local_dataset_path = "/content/sample_dataset.csv"

# Ensure the dataset exists
if not os.path.exists(local_dataset_path):
    raise FileNotFoundError(f"Dataset not found at {local_dataset_path}.")

# Load the local dataset
df = pd.read_csv(local_dataset_path)
print("Dataset loaded successfully!")
print(f"Total columns found: {list(df.columns)}")
print(f"Dataset shape: {df.shape}")

# Preparing the Product Description
product_description = []
product_description_len = []

# Handle column naming variations dynamically
cols = {c.upper(): c for c in df.columns}
title_col = cols.get("TITLE", "TITLE" if "TITLE" in df.columns else df.columns[2])
desc_col = cols.get("DESCRIPTION", "DESCRIPTION" if "DESCRIPTION" in df.columns else df.columns[4])

for idx, row in df.iterrows():
    product = ""
    title = row[title_col]
    if pd.notna(title):
        product += "Title\n" + str(title) + "\n"
    description = row[desc_col]
    if pd.notna(description):
        product += "Description\n" + str(description) + "\n"

    if title or description:
        product = product.strip()
        product_description.append(product)
        product_description_len.append(len(product))

# Chunking the Data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
documents = text_splitter.create_documents(product_description)
print(f"Created {len(documents)} document chunks.")

# Create an embedding model using LangChain
embeddings = OpenAIEmbeddings()

# Create a vector store using the created chunks and the embeddings model
vector = FAISS.from_documents(documents, embeddings)
print("Vector store built successfully!")

# Create ChatOpenAI object
llm = ChatOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Create prompt and retriever
output_parser = StrOutputParser()
prompt = ChatPromptTemplate.from_template("""Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}""", output_parser = output_parser)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vector.as_retriever()

# Build the RAG chain using LCEL
retrieval_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG Retrieval Chain built and ready to use!")

In [ ]:
!pip install faiss-cpu

In [ ]:
# Verification of clean imports after library reinstalls
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_community.vectorstores import FAISS
    print("✅ All imports are fully resolved and successful!")
except ImportError as e:
    print(f"❌ Import verification failed: {e}")

In [ ]:
!pip install -U langchain-community

In [ ]:
# Re-verify that all imports are now resolved
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_community.vectorstores import FAISS
    print("✅ All imports are fully resolved and successful!")
except ImportError as e:
    print(f"❌ Import verification failed: {e}")

In [ ]:
!pip install langchain-community

In [ ]:
import pandas as pd

df_check = pd.read_csv('/content/sample_dataset.csv')
print(f"DataFrame Shape: {df_check.shape}")
print("Columns:", list(df_check.columns))
display(df_check.head(10))

In [ ]:
from google.colab import files
import os

print("Please select and upload your local 'sample_dataset.csv' file:")
uploaded = files.upload()

# Rename the uploaded file to 'sample_dataset.csv' if it has a different name
for filename in uploaded.keys():
    os.rename(filename, '/content/sample_dataset.csv')
    print(f"\nSuccessfully uploaded and set up {filename} as /content/sample_dataset.csv!")

In [ ]:
import zipfile
import os
import shutil

# Target the raw uploaded ZIP file from the file list
zip_path = "/content/Dataset-20260222T014711Z-1-001.zip"
extract_path = "/content/extracted_dataset"
target_csv_path = "/content/sample_dataset.csv"

if os.path.exists(zip_path) and zipfile.is_zipfile(zip_path):
    print("Extracting dataset archive...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    # Walk through the extracted files to find 'sample_dataset.csv'
    found_csv = None
    for root, dirs, files in os.walk(extract_path):
        for file in files:
            if file.endswith("sample_dataset.csv"):
                found_csv = os.path.join(root, file)
                break

    if found_csv:
        # Replace the target path with the actual extracted CSV
        if os.path.exists(target_csv_path):
            os.remove(target_csv_path)
        shutil.move(found_csv, target_csv_path)
        print("Successfully extracted and set up the actual 'sample_dataset.csv'!")
    else:
        print("Could not find 'sample_dataset.csv' inside the uploaded ZIP archive.")
else:
    print(f"Could not find or open the ZIP file at {zip_path}.")

In [ ]:
import zipfile
import os
import shutil

# Path of the uploaded ZIP file (currently named sample_dataset.csv)
zip_path = "/content/sample_dataset.csv"
extract_path = "/content/extracted_dataset"

if zipfile.is_zipfile(zip_path):
    print("Extracting dataset archive...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    # Walk through the extracted files to find 'sample_dataset.csv'
    found_csv = None
    for root, dirs, files in os.walk(extract_path):
        for file in files:
            if file.endswith("sample_dataset.csv"):
                found_csv = os.path.join(root, file)
                break

    if found_csv:
        # Clean up the original zip name placeholder and replace it with the actual CSV
        os.remove(zip_path)
        shutil.move(found_csv, zip_path)
        print("Successfully extracted and set up the actual 'sample_dataset.csv'!")
    else:
        print("Could not find 'sample_dataset.csv' inside the uploaded ZIP archive. Please verify the archive content.")
else:
    print("The uploaded file is already a raw CSV or not a valid zip archive. No extraction needed.")

In [ ]:
# # Code to mount Google Drive and download/save the sample dataset
# from google.colab import drive
# import urllib.request
# import os

# # 1. Mount Google Drive
# drive.mount('/content/drive')

# # 2. Define target directory on your Google Drive
# drive_dest_dir = "/content/drive/My Drive/IK ML Resources/Multi Agent Systems/Dataset/"
# os.makedirs(drive_dest_dir, exist_ok=True)

# # 3. Path where the dataset should be saved
# dataset_path = os.path.join(drive_dest_dir, "sample_dataset.csv")

# # 4. Download a sample CSV dataset to that location if it doesn't exist yet
# # (Replace this URL with the actual URL of your dataset if you have one online)
# sample_url = "https://raw.githubusercontent.com/datasets/gdp/master/data/gdp.csv"

# try:
#     print(f"Downloading dataset to {dataset_path}...")
#     urllib.request.urlretrieve(sample_url, dataset_path)
#     print("Dataset successfully uploaded/saved to your Google Drive!")
# except Exception as e:
#     print(f"An error occurred: {e}")


import pandas as pd
df = pd.read_csv("/content/sample_dataset.csv")
print(df.head())
print(df.columns.tolist())


In [ ]:
product_description = (
    "Title\n" + df["TITLE"].fillna("") +
    "\nDescription\n" + df["DESCRIPTION"].fillna("")
).tolist()
print("Product Description", product_description[10])

-----------------------------------------------------------------------------
## TODAY'S CONTENT BEGINS HERE
-----------------------------------------------------------------------------

### Limitations of the LLMs

Large Language Models (LLMs) like ChatGPT, GPT-4, and other AI systems are incredibly powerful, but they have a key limitation: they rely on pre-trained knowledge, which becomes outdated over time. This is where Tavily plays a crucial role by integrating real-time web search capabilities to keep AI responses up-to-date and contextually relevant.

#### How does Tavily improves it?
- Tavily enables live searches to retrieve the latest news, research papers, and real-world updates.
- LLMs have a knowledge cutoff (e.g., ChatGPT may not know recent events). **Tavily enables AI to bridge this gap by fetching real-time data from the web.**

So, let's set up the API key for using this in our project.
It's FREE!!!!!!

### Set up Tavily Key

In [ ]:
# Setting up the Tavily API Key to do web search
import os
import getpass

os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter Tavily API Key: ")
# os.environ["TAVILY_API_KEY"] = getpass.getpass()

### TaviliySearchResults Tool

`search_tavily(query: str)`:
This function initializes TavilySearchResults with the following parameters & returns the results for the user query:

    - `max_results=5`: Limits the number of search results to 5.
    - `include_answer=True`: Retrieves direct answers when available.
    - `include_raw_content=True`: Fetches full raw text content from the search results.
    - `include_images=True`: Includes images found in the search results.


### Example:
If the chatbot needs real-time information (e.g., latest stock prices, news, or research articles), it can call TavilySearch to retrieve fresh data from the web.
**Example Query:**
- User: "What are the latest AI trends in 2025?"
- AI (using TavilySearch) → Fetches 5 recent web search results and summarizes them in the response.



In [ ]:
%pip install -U langchain-community

In [ ]:
# Import from LangChain for Tavily integration
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

@tool
def search_tavily(query: str):
    """
    Executes a web search using the TavilySearchResults tool.

    Parameters:
        query (str): The search query entered by the user.

    Returns:
        list: A list of search results containing answers, raw content, and images.
    """
    # Create an instance of TavilySearchResults with customized parameters
    search_tool = TavilySearchResults(
        max_results=4,              # Retrieves up to 4 search results
        include_answer=True,        # Includes direct answers when available
        include_raw_content=True,   # Includes full raw text content from search results
        include_images=True,        # Includes images from the search results
    )

    # Invoke search with the given query and return the results
    return search_tool.invoke(query)


### Amazon Product Search Tool

- Importing `create_retriever_tool`:

`create_retriever_tool` is a utility function in LangChain that allows you to wrap a retriever (a component that fetches relevant information from a data source) into a tool that can be used within an AI workflow.

- Creating the `retriever_tool`:
    - **retriever:** This is a retriever instance (coded in the previous class).
    - **name="amazon_search":** Defines the tool's name, which helps identify it when used in multi-tool environments.
    - **description:** Provides a natural language description of what the tool does, which helps AI models understand when to invoke it.

In [ ]:
# import the 'tool' decorator
from langchain.tools import tool
from langchain_core.tools.retriever import create_retriever_tool

# Define a tool for Amazon product search using @tool decorator
@tool
def amazon_product_search(query: str):
    """
    Search for information about Amazon products.
    For any questions related to Amazon products, this tool must be used."""

    # Create the retriever tool
    retriever_tool = create_retriever_tool(
        retriever,  # Retriever object that fetches Amazon product data
        name="amazon_search",
        description="Search for information about Amazon products."
    )

    # Execute search based on query
    return retriever_tool.invoke(query)

### List of tools with Tavily Search and Amazon Product Search

In [ ]:
## List of tools: retriever_tool and search_tool
tools = [search_tavily, amazon_product_search]

### Imports for creating agents

In [ ]:
%pip install -U langchain-openai

In [ ]:
%pip install --upgrade --force-reinstall langchain-core langchain-openai langchain-community

In [ ]:
# Perform necessary imports for creating agents

from langgraph.prebuilt import create_react_agent

from langchain_community.chat_message_histories import ChatMessageHistory  # Used to store chat history in memory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import OpenAI

### Prompt Template for ReAct style conversational agents

- **Creating a ReACT-style Prompt Template**

 ReAct (Reasoning + Acting) prompts define how an AI agent should:
 * Reason about whether to use a tool
 * Format tool calls (Action + Action Input)
 * Process observations from tool results
 * Provide a final answer when no more tools are needed
 * Input variables:

   - tools: List of available tool descriptions
   - tool_names: Names of available tools
   - chat_history: Previous conversation for context
   - input: Current user query
   - agent_scratchpad: Intermediate reasoning steps
    
- **Initializing ChatMessageHistory for Memory**
* `ChatMessageHistory` is a memory store that keeps track of conversation history.
* The `session_id="test-session"` ensures that chat messages are stored for this specific session.
* This helps in maintaining context across user interactions, allowing AI models to provide context-aware responses.

* If you're building a chatbot using LangChain, storing chat history ensures the AI remembers past messages.

**Example scenario:**

    - User: "What’s the latest AI research?"
    - AI: (Fetches real-time research using Tavily and provides an answer.)
    - User: "Summarize it in simpler terms."

The AI, using memory, understands the second request is related to the first and generates a summary.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics.

TOOLS:
------

Assistant has access to the following tools:

{tools}

To use a tool, please use the following format:
```
Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
```

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:
```
Thought: Do I need to use a tool? No
Final Answer: [your response here]
```

Begin!

Previous conversation history:
{chat_history}

New input: {input}
{agent_scratchpad}""",
    input_variables=["agent_scratchpad", "chat_history", "input", "tool_names", "tools"]
)

print('Prompt ready!')
print('Input variables:', prompt.input_variables)
print('\nPrompt Template:', prompt.template)




### Init OpenAI LLM, Create ReAct Agent, Init Agent Executor

1. Initialize the OpenAI Language Model (LLM):
 - OpenAI's completion-based model is used to generate responses.
 - Setting `temperature=0` ensures deterministic and factual responses.
 - A lower temperature reduces randomness, making responses more structured.

2. Creating the ReAct Agent:
 - The agent follows the ReAct framework (Reasoning + Acting).
 - It first thinks through the query and then decides whether to call a tool.
 - If a tool is required, it invokes the appropriate one before responding.

3. Initializing the Agent Executor:
 - The AgentExecutor handles query execution and tool invocation.
 - It decides when the agent should call a tool vs. answer directly.
 - The verbose mode provides detailed logs to track agent reasoning and tool interactions.


#### IK ClassVersion uses CompletionModel instead of ChatModel, but this notebook no longer uses it, why?


##### Legacy ReAct Implementations Used Completion Models

**Historical Context:**
Early ReAct agent implementations often used `OpenAI` (completion-based) instead of `ChatOpenAI` because they were built when **completion models were the primary interface** for LLMs. These implementations prioritized:

- **Direct prompt control**: Completion models allowed exact control over prompt formatting without message structure constraints
- **Simpler parsing**: Raw text outputs were easier to parse for ReAct's specific "Thought → Action → Observation" patterns
- **Established patterns**: Early agent frameworks were designed around completion-style interfaces

**The Trade-offs:**
While completion models offered more granular control over prompt structure, they lacked the built-in capabilities that modern chat models provide, such as native function calling and structured output handling.

**Today's Best Practices:**
Today's agent implementations typically use `ChatOpenAI` and similar chat models because they offer:
- Superior function calling capabilities
- Better tool integration
- More reliable structured outputs
- Built-in message handling that actually enhances multi-step reasoning

**Key Takeaway:**
The choice between completion and chat models was largely a product of when the code was written and what tools were available. Modern ReAct implementations benefit significantly from chat models' advanced capabilities, making the historical preference for completion models largely obsolete.



#### Setup Anthropic Key

In [ ]:
!pip install anthropic langchain-anthropic -U

In [ ]:
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass()

Using OpenAI API key and model gpt4.0 instead of anthropic sonnet. Skip anthropic connection and test open ai connection

In [ ]:
import anthropic
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
# List available models to see what you have access to
try:
    print("Listing available models:")
    available_models = client.models.list()
    for model in available_models.data:
        print(f"Model ID: {model.id}")

    # 6. Test a simple completion with the first available model
    if available_models.data:
        test_model = available_models.data[0].id
        print(f"\nTesting a simple completion with model: {test_model}")

        message = client.messages.create(
            model=test_model,
            max_tokens=100,
            messages=[
                {"role": "user", "content": "Hello, how are you?"}
            ]
        )

        print(f"Response: {message.content[0].text}")

        # 7. If we get here, the direct API connection works
        print("\nDirect Anthropic API connection successful!")
except Exception as e:
    print(f"Error testing Anthropic API: {str(e)}")
    print("Please check your API key and account status.")


In [ ]:

import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("OpenAI API key configured!")


In [ ]:

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

response = llm.invoke("Hello, how are you?")
print(response.content)


In [ ]:
import os
#import anthropic
#from langchain_anthropic import ChatAnthropic
import openai
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.outputs import Generation, LLMResult
from typing import Dict, Any, List, Optional
from langchain_core.runnables import Runnable

import warnings

# Suppress all deprecation warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

class AgentExecutor(Runnable):
    """Wrapper to mimic older AgentExecutor interface but using LangGraph agent."""

    def __init__(
        self,
        agent,
        tools=None,
        verbose=False,
        handle_parsing_errors=True,
        max_iterations=5,
        early_stopping_method="force",
        return_intermediate_steps=False,
        get_session_history=None,
        **kwargs
    ):
        self.agent = agent
        self.tools = tools
        self.verbose = verbose
        self.handle_parsing_errors = handle_parsing_errors
        self.max_iterations = max_iterations
        self.early_stopping_method = early_stopping_method
        self.return_intermediate_steps = return_intermediate_steps
        self.get_session_history = get_session_history  # ← Store function

    def invoke(self, input_dict, config=None, **kwargs):
        query = input_dict.get("input", "")
        chat_history = input_dict.get("chat_history", [])

        # Get session_id from config
        session_id = None
        if config and "configurable" in config:
            session_id = config["configurable"].get("session_id")

        if self.verbose:
            print(f"\n{'='*60}")
            print(f"Processing: {query}")
            print(f"Chat history length: {len(chat_history)}")
            print(f"{'='*60}")

        # Initialize variables
        final_message = None
        tools_used = []

        try:
            messages = list(chat_history) + [HumanMessage(content=query)]
            result = self.agent.invoke({"messages": messages})

            # Track tool usage
            for msg in result["messages"]:
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    for tc in msg.tool_calls:
                        tools_used.append(tc['name'])

            if self.verbose and tools_used:
                print(f"\n Tools used: {', '.join(tools_used)}")

            final_message = result["messages"][-1].content

            if self.verbose:
                print(f"\n{'='*60}")
                print("Agent completed")
                print(f"{'='*60}")

            # Auto-save to session history using the SAME function
            if session_id and self.get_session_history:
                history = self.get_session_history(session_id)
                history.add_user_message(query)
                history.add_ai_message(final_message or "")

            return {
                "output": final_message if final_message else "No response generated.",
                "tools_used": tools_used
            }

        except Exception as e:
            if self.handle_parsing_errors:
                if session_id and self.get_session_history:
                    history = self.get_session_history(session_id)
                    history.add_user_message(query)
                    history.add_ai_message(f"Error: {str(e)}")

                return {"output": f"Error: {str(e)}", "tools_used": tools_used}
            raise


# Use ChatAnthropic
# llm_model = ChatAnthropic(
#     model="claude-sonnet-4-6",
#     temperature=0,
#     anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
# )


# Use chatopenai instead of ChatAnthropic
llm_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)



# Create ReAct agent using the LLM, tools
# This ReAct agent (from langgraph.prebuilt) implicitly used the ReAct prompt template,
# so there is no need to use our ReAct-style prompt
agent = create_react_agent(
    model=llm_model,  # Anthropic model
    tools=tools,      # A list of external tools (web search, product retrieval)
    #prompt=prompt     # The ReAct-style prompt guiding the agent's thought process
)

# Session store for chat histories
session_store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# Initialize the Agent Executor to run the ReAct agent with some error handling
agent_executor = AgentExecutor(
    agent=agent,    # The ReAct agent created in the previous step
    tools=tools,    # Ensures the agent has access to the required tools
    verbose=True,   # Enables detailed logging for debugging and tracking execution steps
    handle_parsing_errors=True,  # Handles parsing errors during agent execution
    max_iterations=5,            # Set a reasonable maximum number of iterations
    early_stopping_method="force",  # Force stop if max_iterations reached
    return_intermediate_steps=False,
    get_session_history=get_session_history
)

**NOTE: AgentExecutor:**

Purpose: AgentExecutor manages the process of an agent deciding what actions to take, taking those actions, observing the results, and repeating until the task is complete.

**Key functionality:**

Takes an agent and a set of tools as input
* Manages the loop of agent execution (decision → action → observation)
* Handles error cases and edge conditions
* Manages the termination of the agent process


**Basic flow:**

* The executor receives a user query
* It passes this to the agent, which decides what tool to use
* The executor runs the selected tool with the provided input
* It provides the tool's output back to the agent
* This cycle continues until the agent decides it has a final answer

**NOTE: max_iterations**:

Defines the maximum number of cycles the agent will be allowed to execute before being forced to stop, regardless of whether it has completed its task.

Each iteration represents one full cycle of:

* The agent deciding what action to take
* The executor running the selected tool
* The agent receiving the results of that action

**Purpose:**

* Preventing infinite loops: It stops agents that might get stuck in circular reasoning or repetitive actions.
* Resource management: It helps control computational resources by ensuring the agent doesn't run for too long.
* Cost control: Since many LLM calls are made during agent execution, this parameter helps manage API costs.

**Key Functionality:**
* By default, LangChain typically sets max_iterations to a value like 15, but you can adjust it based on your specific use case. For complex tasks that might require more steps, you could increase this value, while for simpler tasks, you might decrease it.

* If the agent reaches the maximum number of iterations without completing the task, the AgentExecutor will typically return whatever partial result or state the agent has reached, along with a notification that the maximum iterations were exceeded.

#### Initialize Agent with Chat History

- Initializing an Agent with chat history:
 - `session_id` is useful in real-world applications where session tracking is required. Here, it simply returns the in-memory chat history for simplicity.
 - `RunnableWithMessageHistory` ensures the agent retains context across multiple interactions. This is useful for multi-turn conversations where past messages influence future responses.


In [ ]:
# Initialize an Agent with Chat History to maintain conversation context
agent_with_chat_history = RunnableWithMessageHistory(
  agent_executor,                       # Main AgentExecutor responsible for executing queries and invoking tools.
  get_session_history,                  # Function to get chat history based on session ID
  input_messages_key="input",           # Specifies the key in the input dictionary where the user query is stored.
  history_messages_key="chat_history",  # Specifies the key under which the conversation history is stored.
)

#### Query1: Try a query based on latest data and check whether Tavily Search is triggered

In [ ]:
# Invoke the agent with chat history

# Clear session
session_id = "test"
if session_id in session_store:
    session_store[session_id].clear()

# Let's try a query based on recent data and check whether 'Tavily Search' is triggered
response1 = agent_with_chat_history.invoke(
    {"input": "Latest fashion trends in 2025?"},
    config={"configurable": {"session_id": session_id}}
)


print("\n\nResponse to first query <What are the latest fashion trends in 2025?>:")
print(response1["output"])

# Check tool usage
print("\n\nTool Usage Check:")
print("="*60)
tools_used = response1.get("tools_used", [])
if tools_used:
    for tool in tools_used:
        print(f"✅ Tool Called: {tool}")
    if any('tavily' in t.lower() for t in tools_used):
        print("\n✅ Tavily Search WAS used!")
else:
    print("❌ No tools were used")
print("="*60)

In [ ]:
print(f"Chat history in session: {session_id}")
history = get_session_history(session_id)
print(f"Messages in history: {len(history.messages)}")
for i, msg in enumerate(history.messages):
    print(f"  {i+1}. {msg.__class__.__name__}: {str(msg.content)[:100]}...")

#### Query2: Try a query based on Amazon Product data and check if Amazon Product Search Tool is triggered

In [ ]:
# Invoke the agent with chat history from Query1

# Let's try a query based on amazon data & check if 'amazon_search' tool is triggered
response2 = agent_with_chat_history.invoke(
    {"input": "Does Amazon have these latest trends?"},
    config={"configurable": {"session_id": session_id}},
)


print("\n\nResponse to second query <Does Amazon have these latest trends?>:")
print(response2["output"])

# Check tool usage
print("\n\nTool Usage Check:")
print("="*60)
tools_used = response2.get("tools_used", [])
if tools_used:
    for tool in tools_used:
        print(f"✅ Tool Called: {tool}")
    if any('amazon' in t.lower() for t in tools_used):
        print("\n✅ Amazon Search WAS used!")
else:
    print("❌ No tools were used")
print("="*60)


#### Query3: Try a query based on previous history

In [ ]:
# Invoke the agent with chat history
# Let's try a query based on previous history
response3 = agent_with_chat_history.invoke(
    {"input": "Suggest me some best fashion accessories from Amazon "},
    config={"configurable": {"session_id": session_id}},
)

print("\n\nResponse to third query <Suggest me some best fashion accessories from Amazon>:")
print(response3['output'])

#### UI for chatbot with agents

##### With explicit new session id on app start

In [ ]:
# Building a UI for the chatbot with agents
import gradio as gr

# Initialize session-based chat history
session_store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    """Fetch or create a chat history instance for a given session."""
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# Initialize Agent Executor with get_session_history function
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
    early_stopping_method="force",
    return_intermediate_steps=False,
    get_session_history=get_session_history
)

# Wrap agent with session-based chat history
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# Define function for Gradio interface
def chat_with_agent(user_input, session_id="default_session"):
    """Processes user input and maintains session-based chat history."""

    if not user_input.strip():
        return "Please enter a query."

    # Invoke the agent with session memory
    response = agent_with_chat_history.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    )

    # Extract output and tools used
    if isinstance(response, dict):
        output = response.get("output", "No response generated.")
        tools_used = response.get("tools_used", [])

        # Optionally append tools info
        if tools_used:
            output += f"\n\n Tools used: {', '.join(tools_used)}"

        return output
    else:
        return "Error: Unexpected response format"

# Create Gradio app interface
with gr.Blocks() as app:
    gr.Markdown("# 🤖 Review Genie - Agents & ReAct Framework")
    gr.Markdown("Enter your query below and get AI-powered responses with session memory.")

    with gr.Row():
        with gr.Column():
            session_box = gr.Textbox(
                label="Session ID:",
                value="default_session",
                placeholder="Enter session ID for memory persistence"
            )
            input_box = gr.Textbox(
                label="Enter your query:",
                placeholder="Ask something...",
                lines=3
            )
            submit_button = gr.Button("Submit", variant="primary")
            clear_button = gr.Button("Clear History")

        with gr.Column():
            output_box = gr.Textbox(label="Response:", lines=15)

    def clear_session_history(session_id):
        """Clear chat history for a session."""
        if session_id in session_store:
            session_store[session_id].clear()
        return f"History cleared for session: {session_id}"

    submit_button.click(
        chat_with_agent,
        inputs=[input_box, session_box],
        outputs=output_box
    )

    clear_button.click(
        clear_session_history,
        inputs=session_box,
        outputs=output_box
    )

# Launch the Gradio app
app.launch(debug=False, share=False)

### Adding more features to the chatbot......


#### Multi-Tool Agent

**Uses create_react_agent:** Simpler way to create an agent. It's a higher-level abstraction that is extensible to create multiple agents





In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Split product descriptions into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20
)

documents = splitter.create_documents(product_description)

# Convert text into searchable vectors
embeddings = OpenAIEmbeddings()
vector = FAISS.from_documents(documents, embeddings)

# Create the retriever
retriever = vector.as_retriever()

print("Retriever is ready!")


In [ ]:
# Create a Multi-Tool Agent
from langgraph.prebuilt import create_react_agent
# from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


tools = [search_tavily, amazon_product_search]

# chat_model = ChatAnthropic(
#     model="claude-sonnet-4-20250514",
#     temperature=0,
#     anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
# )

chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# Create agent (replaces initialize_agent)
multi_tool_agent = create_react_agent(
    model=chat_model,
    tools=tools
)

# Test the multi-tool-agent system
query = "Which one is recommended - MacBook Air or Dell XPS laptop?"
result = multi_tool_agent.invoke({"messages": [HumanMessage(content=query)]})
response = result["messages"][-1].content
print(response)

#### Multi-Tool Agent with Few shot examples

In [ ]:
import inspect
from langgraph.prebuilt import create_react_agent
print(inspect.signature(create_react_agent))

In [ ]:
# Create a Multi-Tool Agent with Few-Shot Examples
# ------------------------------------------------
# Uses create_react_agent from LangGraph with prompt template for few-shot prompting.
# This replaces the older and deprecated create_react_agent + AgentExecutor from LangChain that was used in the IK version.

from langgraph.prebuilt import create_react_agent
# from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import os

#
# Few-shot examples
# examples = [
#     {
#         "question": "Compare iPhone 15 vs Samsung Galaxy S24 pricing",
#         "scratchpad": """Thought: Need to search for both phone prices.
# Action: search_tavily
# Action Input: iPhone 15 Samsung Galaxy S24 price comparison
# Observation: iPhone 15: $799, Galaxy S24: $799, similar features
# Thought: I have the comparison data.
# Final Answer: Both phones start at $799 with comparable flagship features."""
#     },
#     {
#         "question": "Find laptops under $1000",
#         "scratchpad": """Thought: User wants budget laptop options.
# Action: amazon_product_search
# Action Input: laptop under 1000
# Observation: ASUS VivoBook ($699), Acer Aspire 5 ($549), HP Pavilion ($799)
# Thought: Found good options in budget.
# Final Answer: Top options: ASUS VivoBook $699, Acer Aspire 5 $549, HP Pavilion $799."""
#     }
# ]

# # Build few-shot prompt
# tool_names = ["search_tavily", "amazon_product_search"]

# few_shot_prompt = f"""You are a helpful shopping assistant.

# Format for answering:
# Thought: [your reasoning]
# Action: [tool name, one of: {', '.join(tool_names)}]
# Action Input: [input to the tool]
# Observation: [tool result]
# ... (repeat as needed)
# Thought: I have enough information.
# Final Answer: [your final response]

# Examples:

# Question: {examples[0]['question']}
# {examples[0]['scratchpad']}

# Question: {examples[1]['question']}
# {examples[1]['scratchpad']}

# Now handle the user's question. Be thorough and provide specific details."""

# # Using openai instead of ChatAnthropic
# # chat_model = ChatAnthropic(
# #     model="claude-sonnet-4-20250514",
# #     temperature=0,
# #     anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
# # )


# chat_model = ChatOpenAI(
#     model="gpt-4o-mini",
#     temperature=0
# )


# # Create tools
# tools = [search_tavily, amazon_product_search]

# # Create agent with few-shot examples
# multi_tool_agent = create_react_agent(
#     model=chat_model,
#     tools=tools,
#     prompt=few_shot_prompt
# )

# Test the agent
query = "Compare MacBook Air and Dell XPS laptops."
#result = multi_tool_agent.invoke({"messages": [HumanMessage(content=query)]})
print(result["messages"][-1].content)


In [ ]:
# Check which tools were invoked
print("\n\nTool Usage:")
print("="*60)
tools_used = []
for msg in result["messages"]:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            tools_used.append(tc['name'])
            print(f"   Tool: {tc['name']}")
            print(f"   Input: {tc['args']}")

if not tools_used:
    print("❌ No tools were used")
else:
    print(f"\n✅ Tools used: {', '.join(tools_used)}")
print("="*60)

#### Moving toward Multi-Agent systems

In [ ]:
# Multi-Agent System with Coordinator
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
import os

# Initialize model
chat_model = ChatOpenAI(
    model="gpt-4o-mini",  # or "gpt-4o", "gpt-3.5-turbo"
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create specialized agents
search_agent = create_react_agent(model=chat_model, tools=[search_tavily])
product_agent = create_react_agent(model=chat_model, tools=[amazon_product_search])

# Wrap agents as tools
@tool
def SearchAgent(query: str) -> str:
    """Useful for searching the web for information, trends, news."""
    result = search_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content

@tool
def ProductAgent(query: str) -> str:
    """Useful for finding products on Amazon."""
    result = product_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content

# Create coordinator agent
coordinator_agent = create_react_agent(
    model=chat_model,
    tools=[SearchAgent, ProductAgent]
)

# Test
query = "I'm traveling to Paris next month and want to dress fashionably. What are the latest fashion trends in Paris this season, and are these available on Amazon?"
result = coordinator_agent.invoke({"messages": [HumanMessage(content=query)]}, config={"recursion_limit": 10})
print(result["messages"][-1].content)

#### Another Multi Agent System to handle get_weather



In [ ]:
import requests
import getpass

# OpenWeatherMap API Configuration
API_KEY = getpass.getpass("Enter OpenWeatherMap API Key: ")
BASE_URL = "http://api.openweathermap.org/data/2.5/weather"

@tool
def get_weather(city: str) -> str:
    """
    Fetches CURRENT real-time weather data for a given city.
    Note: This tool only provides current weather, not forecasts.
    """
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        return f"Error fetching weather data: {response.json().get('message', 'Unknown error')}"

    # Extract weather details
    data = response.json()
    temp = data["main"]["temp"]
    condition = data["weather"][0]["main"].lower()
    humidity = data["main"]["humidity"]

    return f"Weather in {city}: {temp}°C, {condition}, humidity {humidity}%"


# Multi-Tool Agent with get_weather added
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

tools = [search_tavily, amazon_product_search, get_weather]

multi_tool_agent = create_react_agent(
    model=chat_model,
    tools=tools
)

# Test the agent
query = "I will travel to Paris now. What all should I carry to suit the weather conditions there? Should I purchase any from Amazon?"
result = multi_tool_agent.invoke({"messages": [HumanMessage(content=query)]})
print(result["messages"][-1].content)

In [ ]:
# Multi-Agent with weather_agent, search_agent and product_agent
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# Create specialized agents
weather_agent = create_react_agent(model=chat_model, tools=[get_weather])
search_agent = create_react_agent(model=chat_model, tools=[search_tavily])
product_agent = create_react_agent(model=chat_model, tools=[amazon_product_search])

# Wrap agents as tools
@tool
def WeatherAgent(query: str) -> str:
    """Useful for getting current weather information for a city."""
    result = weather_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content

@tool
def SearchAgent(query: str) -> str:
    """Useful for searching the web for information, trends, news."""
    result = search_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content

@tool
def ProductAgent(query: str) -> str:
    """Useful for finding products on Amazon."""
    result = product_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content

# Create coordinator agent
new_coordinator_agent = create_react_agent(
    model=chat_model,
    tools=[SearchAgent, ProductAgent, WeatherAgent],
    prompt="""You are a travel planning coordinator.

- WeatherAgent: Use ONLY for current weather conditions
- SearchAgent: Use for future weather forecasts, trends, and general information
- ProductAgent: Use for finding products on Amazon

For queries about future weather (next week, next month, etc.), use SearchAgent to search for typical weather patterns."""
)

# Test
query = "I'm planning a trip to Paris tomorrow. \
        Could you check the typical weather in Paris this week, \
        recommend fashionable outfits appropriate for those weather conditions?"
result = new_coordinator_agent.invoke({"messages": [HumanMessage(content=query)]})
print(result["messages"][-1].content)

# Show agents called
print("\n\n Agents Called:")
print("="*60)
for msg in result["messages"]:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  ✅ {tc['name']}")
print("="*60)



## Conclusion

### What We Built

**Two separate systems:**

**1. Single Agent with Chat History & Gradio UI**
```
┌─────────────────────────────────────────────────────────────────┐
│                      Gradio Web Interface                       │
│                  (Session-based Chat History)                   │
└──────────────────────────────┬──────────────────────────────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   AgentExecutor     │
                    │   (with Memory)     │
                    └──────────┬──────────┘
                               │
                    ┌──────────┴──────────┐
                    │                     │
                    ▼                     ▼
             search_tavily      amazon_product_search
```

**2. Multi-Agent Coordinator System**
```
                    ┌─────────────────────┐
                    │  Coordinator Agent  │
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           │                   │                   │
           ▼                   ▼                   ▼
   ┌───────────────┐   ┌───────────────┐   ┌───────────────┐
   │ WeatherAgent  │   │  SearchAgent  │   │ ProductAgent  │
   └───────┬───────┘   └───────┬───────┘   └───────┬───────┘
           │                   │                   │
           ▼                   ▼                   ▼
      get_weather         search_tavily    amazon_product_search
```

### Key Concepts Covered

| Concept | Description |
|---------|-------------|
| **ReAct Pattern** | Reasoning + Acting loop (Thought → Action → Observation → Answer) |
| **Tool Calling** | LLM decides which tools to use and with what inputs |
| **Few-Shot Prompting** | Providing examples to guide agent behavior |
| **Session Memory** | Chat history persistence using `RunnableWithMessageHistory` |
| **Agent-as-Tool** | Wrapping specialized agents as tools for a coordinator |
| **Multi-Agent Orchestration** | Coordinator delegates tasks to specialized agents |
| **Web Interface** | Interactive Gradio UI for user interaction |

### Current Capabilities

| What It Does | What It Doesn't Do |
|--------------|---------------------|
| Semi-autonomous decisions | Fully autonomous operation |
| Reactive to queries | Proactive actions |
| Specialized agent roles | Self-directed goals |
| Coordinated orchestration | Persistent memory across app restarts |
| Session-based chat history | Long-term memory storage |
| Web-based user interface | Multi-user authentication |

### Code Patterns Used

**1. Single Agent with Session Memory**
```python
# Session store
session_store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# Wrap agent with memory
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)
```

**2. Multi-Agent Coordinator**
```python
# Create specialized agents
search_agent = create_react_agent(model=chat_model, tools=[search_tavily])
product_agent = create_react_agent(model=chat_model, tools=[amazon_product_search])

# Wrap agents as tools
@tool
def SearchAgent(query: str) -> str:
    """Useful for searching the web."""
    result = search_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content

# Create coordinator
coordinator = create_react_agent(
    model=chat_model,
    tools=[SearchAgent, ProductAgent, WeatherAgent]
)
```

### Path Forward: Full Autonomy

To evolve this into a fully autonomous system:

| Feature | Implementation |
|---------|----------------|
| **Continuous operation** | Event loops, scheduled triggers |
| **Persistent memory** | Database-backed chat history (Redis, PostgreSQL) |
| **Goal setting** | Planning agents, task decomposition |
| **Proactive actions** | Monitoring agents, alert systems |
| **Human oversight** | Approval workflows, guardrails |

### Summary

We progressively built two AI agent systems:

1. **Single Agent with Tools** — ReAct agent with search and product tools, session memory, and Gradio UI
2. **Multi-Agent Coordinator** — Specialized agents orchestrated by a supervisor

**This is a foundation for building more advanced AI agent applications.**